In [5]:
import os
import pandas as pd

In [6]:
# path to folder
folder = "rawData"

# store all dataframes in a dictionary
data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(path)

# check loaded files
print(data.keys())


dict_keys(['Coordinates - Grid Cell Center.csv', 'Coordinates - Grid Gehäusewand.csv', 'Coordinates - Grid JR1 Center.csv', 'T_grid_cc_i.csv', 'T_grid_jr1c_i.csv'])


In [ ]:
grid_pairs = [
    ("Cell Center", "Coordinates - Grid Cell Center.csv", "T_grid_cc_i.csv"),
    ("JR1 Center", "Coordinates - Grid JR1 Center.csv", "T_grid_jr1c_i.csv"),
    ("Gehaeusewand", "Coordinates - Grid Gehäusewand.csv", "T_grid_cc_i.csv"),#hier ändern
]

all_frames = []

In [8]:
for grid_name, coord_file, temp_file in grid_pairs:
    if coord_file not in data or temp_file not in data:
        print(f"Skipping {grid_name}: missing {coord_file} or {temp_file}")
        continue

    coord_df = data[coord_file].copy()
    temp_df = data[temp_file].copy()

    temp_cols = [c for c in temp_df.columns if c != "Physical Time (s)"]
    n = min(len(coord_df), len(temp_cols))

    if len(coord_df) != len(temp_cols):
        print(
            f"Warning for {grid_name}: {len(coord_df)} coordinates vs {len(temp_cols)} sensors. Using first {n}."
        )

    coord_df = coord_df.iloc[:n].reset_index(drop=True)
    temp_cols = temp_cols[:n]

    # Turn wide temperature table into long format
    long_temp = temp_df.melt(
        id_vars=["Physical Time (s)"],
        value_vars=temp_cols,
        var_name="Sensor",
        value_name="Temperature"
    )

    # Map each sensor column back to its coordinate row
    sensor_to_idx = {sensor_name: idx for idx, sensor_name in enumerate(temp_cols)}
    long_temp["sensor_idx"] = long_temp["Sensor"].map(sensor_to_idx)

    merged = long_temp.merge(
        coord_df.reset_index().rename(columns={"index": "sensor_idx"}),
        on="sensor_idx",
        how="left"
    )

    merged = merged[[
        "Position[X] (m)",
        "Position[Y] (m)",
        "Position[Z] (m)",
        "Physical Time (s)",
        "Temperature"
    ]]

    all_frames.append(merged)

df_new = pd.concat(all_frames, ignore_index=True)
print(df_new.head())

   Position[X] (m)  Position[Y] (m)  Position[Z] (m)  Physical Time (s)  \
0    -6.847063e-18        -0.098911         0.052177                0.1   
1    -6.847063e-18        -0.098911         0.052177                0.2   
2    -6.847063e-18        -0.098911         0.052177                0.3   
3    -6.847063e-18        -0.098911         0.052177                0.4   
4    -6.847063e-18        -0.098911         0.052177                0.5   

   Temperature  
0    39.998956  
1    39.998010  
2    39.997308  
3    39.996851  
4    39.996729  


In [9]:
# Save as CSV (Excel-friendly in German locale)
df_new.to_csv("processedData/combinedData.csv", index=False, sep=";", encoding="utf-8-sig")

print("CSV created successfully!")

CSV created successfully!
